In [1]:
"""
Main Engine (Clean, Single-File, MIN-HYPERPARAM) + Event Counting (Rising-edge + Refractory)
- Dataset: MHEALTH (.log)
- LOSO per-activity
- Detector: block-level c_full -> robust_z -> steady score s_full, window slices
- Fusion: block-level self-consistency quality -> softmax weights (shared for all windows)
- Calibration: trial-wise quantile calibration (ONLY detector knob)
- Post:
  - fixed hysteresis decode for pseudo (detector) (no knobs exposed)
  - fixed hysteresis decode for model p_hat -> event counting (rising-edge + refractory)
- Model: latent z + transition prob p_hat(t) + reconstruction
- Training: recon + soft pseudo supervision + pair consistency + inertial invariance
- Outputs:
  - results_loso.csv (soft MAE/MSE + count metrics)
  - 2 plots per activity (representative fold, test subject):
    (1) dominant raw magnitude + p_hat/state + event markers
    (2) PCA2D of z(t): steady vs transition separation (test subject)
"""

import os, glob, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy.signal import savgol_filter
import matplotlib.pyplot as plt


# =========================================================
# 0) CONFIG (min)
# =========================================================
CONFIG = {
    # data
    "data_dir": "/content/drive/MyDrive/Colab Notebooks/HAR_data/MHEALTHDATASET",
    "target_activities": [6, 7, 12],
    "fs": 50,

    # windowing
    "window_size": 100,
    "stride": 50,

    # sensors (missing-friendly)
    "groups": ["chest_acc", "ankle_acc", "arm_acc", "ankle_gyro", "arm_gyro"],

    # training
    "batch_size": 64,
    "epochs": 20,
    "lr": 1e-3,
    "seed": 42,

    # model
    "latent_dim": 64,
    "hidden_dim": 128,

    # losses (4 lambdas only)
    "lambda_recon": 1.0,
    "lambda_soft": 0.5,
    "lambda_pair": 0.3,
    "lambda_inertial": 0.2,

    # ONLY detector knob kept (trial-wise calibration width)
    "calib_q": 0.2,

    # output
    "out_dir": "./out_main_engine",
    "plot_sec": 60,  # for plot display length (seconds)

    # counting (kept fixed; not exposed as tunable "many params")
    "count_refractory_sec": 0.40,   # refractory duration
    "model_state_q_enter": 0.90,
    "model_state_q_exit": 0.10,
}

# =========================================================
# 0.1) GT COUNT (given)
# =========================================================
GT_COUNT = {
    6: {
        "subject1": 21, "subject2": 19, "subject3": 21, "subject4": 20, "subject5": 20,
        "subject6": 20, "subject7": 20, "subject8": 21, "subject9": 21, "subject10": 20,
    },
    7: {
        "subject1": 20, "subject2": 20, "subject3": 20, "subject4": 20, "subject5": 20,
        "subject6": 20, "subject7": 20, "subject8": 19, "subject9": 19, "subject10": 20,
    },
    12: {
        "subject1": 20, "subject2": 22, "subject3": 21, "subject4": 21, "subject5": 20,
        "subject6": 21, "subject7": 19, "subject8": 20, "subject9": 20, "subject10": 20,
    },
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CONFIG["out_dir"], exist_ok=True)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CONFIG["seed"])


# =========================================================
# 1) MHEALTH LOADING
# =========================================================
def load_mhealth_df(data_dir: str, target_activities):
    log_files = glob.glob(os.path.join(data_dir, "*.log"))
    if not log_files:
        raise FileNotFoundError(f"No .log files found in {data_dir}")

    all_rows = []
    for fpath in log_files:
        filename = os.path.basename(fpath)
        try:
            sub_id = int("".join(filter(str.isdigit, filename)))
        except:
            sub_id = 0

        try:
            df = pd.read_csv(fpath, sep=r"\s+", header=None, engine="python")
        except:
            df = pd.read_csv(fpath, sep="\t", header=None)

        if df.shape[1] <= 23:
            raise ValueError("Unexpected MHEALTH column count.")

        df = df.copy()
        df["label"] = df.iloc[:, 23].astype(int)
        df["subject_id"] = sub_id
        df = df[df["label"].isin(target_activities)].copy()
        if len(df) > 0:
            all_rows.append(df)

    if not all_rows:
        raise ValueError("No data found for specified activities.")

    df_all = pd.concat(all_rows, ignore_index=True)
    print(
        f"[Data] Total samples: {len(df_all)} | Subjects: {sorted(df_all['subject_id'].unique())} | Acts: {sorted(df_all['label'].unique())}"
    )
    return df_all


def get_group_array_from_block(block_np: np.ndarray, group: str):
    if group == "chest_acc":
        return block_np[:, 0:3]
    if group == "ankle_acc":
        return block_np[:, 5:8]
    if group == "arm_acc":
        return block_np[:, 14:17]
    if group == "ankle_gyro":
        return block_np[:, 8:11]
    if group == "arm_gyro":
        return block_np[:, 17:20]
    return None


def create_blocks_by_subject_activity(df_all: pd.DataFrame, win_size: int):
    """
    NOTE:
    This constructs raw blocks by concatenating all samples with the same label for a subject.
    If you want only one contiguous segment (e.g., longest segment), you would need a different splitter.
    Here we keep your original design.
    """
    blocks = []
    base_cols = list(range(24))
    for sub in sorted(df_all["subject_id"].unique()):
        sub_df = df_all[df_all["subject_id"] == sub]
        for act in sorted(sub_df["label"].unique()):
            act_df = sub_df[sub_df["label"] == act]
            raw = act_df[base_cols].to_numpy(dtype=np.float32)
            if len(raw) >= win_size:
                blocks.append({"subject": int(sub), "act": int(act), "raw": raw})
    return blocks


# =========================================================
# 2) Detector + Fusion (fixed internals)
# =========================================================
def robust_zscore(x: np.ndarray, eps=1e-6, mad_floor=1e-3):
    med = np.median(x)
    mad = np.median(np.abs(x - med)) + eps
    mad = max(float(mad), float(mad_floor))
    return (x - med) / mad


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def acc_change_score(acc_3: np.ndarray, var_win: int = 10):
    T = len(acc_3)
    u = acc_3 / (np.linalg.norm(acc_3, axis=1, keepdims=True) + 1e-8)

    cos_prev = np.sum(u[1:] * u[:-1], axis=1)
    cos_prev = np.clip(cos_prev, -1.0, 1.0)

    dir_change = np.zeros(T, dtype=np.float32)
    dir_change[1:] = 1.0 - cos_prev

    half = var_win // 2
    dir_var = np.zeros(T, dtype=np.float32)
    for i in range(T):
        s = max(0, i - half)
        e = min(T, i + half + 1)
        local = u[s:e]
        dir_var[i] = float(np.mean(np.var(local, axis=0)))

    return (dir_change + dir_var).astype(np.float32)


def gyro_change_score(gyro_3: np.ndarray, var_win: int = 10):
    T = len(gyro_3)
    mag = np.linalg.norm(gyro_3, axis=1)

    dmag = np.zeros(T, dtype=np.float32)
    dmag[1:] = np.abs(mag[1:] - mag[:-1])

    half = var_win // 2
    lvar = np.zeros(T, dtype=np.float32)
    for i in range(T):
        s = max(0, i - half)
        e = min(T, i + half + 1)
        lvar[i] = float(np.var(mag[s:e]))

    return (dmag + lvar).astype(np.float32)


def compute_group_change(group_key: str, Xg: np.ndarray):
    if Xg is None:
        return None
    if "acc" in group_key:
        return acc_change_score(Xg, var_win=10)
    if "gyro" in group_key:
        return gyro_change_score(Xg, var_win=10)
    return np.linalg.norm(np.diff(Xg, axis=0, prepend=Xg[:1]), axis=1).astype(np.float32)


def steady_score_from_change(c: np.ndarray):
    cz = robust_zscore(c, eps=1e-6, mad_floor=1e-3)
    b = np.percentile(cz, 80)
    y = sigmoid((cz - b) / 0.5)     # transition prob
    s = 1.0 - y                    # steady score

    if len(s) >= 11:
        s = savgol_filter(s, window_length=11, polyorder=2).astype(np.float32)
        s = np.clip(s, 0.0, 1.0)
    return s.astype(np.float32)


def cosine_sim(a: np.ndarray, b: np.ndarray, eps=1e-8):
    na = np.linalg.norm(a) + eps
    nb = np.linalg.norm(b) + eps
    return float(np.dot(a, b) / (na * nb))


def estimate_self_consistency_quality(group_key: str, X_full: np.ndarray, s_full: np.ndarray):
    q = 0.2
    k_pairs = 128

    T = len(s_full)
    order = np.argsort(s_full)
    n = max(5, int(q * T))
    low_idx = order[:n]
    high_idx = order[-n:]
    if len(low_idx) < 5 or len(high_idx) < 5:
        return 0.0

    if "acc" in group_key:
        V = X_full / (np.linalg.norm(X_full, axis=1, keepdims=True) + 1e-8)
    else:
        V = X_full

    def mean_pair_sim(idxs):
        sims = []
        for _ in range(k_pairs):
            i, j = np.random.choice(idxs, size=2, replace=True)
            sims.append(cosine_sim(V[i], V[j]))
        return float(np.mean(sims))

    S_high = mean_pair_sim(high_idx)
    S_low = mean_pair_sim(low_idx)
    return float(S_high - S_low)


def fuse_group_scores_block(groups_dict):
    avail = [(g, d) for g, d in groups_dict.items() if d["s_full"] is not None]
    if not avail:
        raise ValueError("No available groups for fusion.")

    qs = np.array([d["q_full"] for _, d in avail], dtype=np.float32)
    tau = 0.25
    ws = np.exp(qs / max(tau, 1e-6))
    ws = ws / (ws.sum() + 1e-8)

    T = len(avail[0][1]["s_full"])
    s_fused = np.zeros(T, dtype=np.float32)
    weights, qualities = {}, {}
    for (g, d), w in zip(avail, ws):
        s_fused += float(w) * d["s_full"]
        weights[g] = float(w)
        qualities[g] = float(d["q_full"])

    s_fused = np.clip(s_fused, 0.0, 1.0)
    return s_fused, weights, qualities


def quantile_calibrate_01(y: np.ndarray, q: float, eps: float = 1e-6):
    q = float(np.clip(q, 0.0, 0.49))
    lo = float(np.quantile(y, q))
    hi = float(np.quantile(y, 1.0 - q))
    denom = max(hi - lo, eps)
    y_cal = (y - lo) / denom
    return np.clip(y_cal, 0.0, 1.0).astype(np.float32)


def hysteresis_decode(y01: np.ndarray, enter: float = 0.7, exit: float = 0.3):
    state = np.zeros_like(y01, dtype=np.int32)
    on = 0
    for i, v in enumerate(y01):
        if on == 0:
            if v >= enter:
                on = 1
        else:
            if v <= exit:
                on = 0
        state[i] = on
    return state.astype(np.float32)


# =========================================================
# 2.1) Event counting: rising-edge + refractory
# =========================================================
def rising_edge_events_with_refractory(state01: np.ndarray, fs: int, refractory_sec: float):
    """
    state01: (T,) float/int in {0,1}
    Event = rising edge (0->1), then lockout for refractory_sec.
    Returns: (pred_count, event_indices)
    """
    s = (state01 > 0.5).astype(np.int32)
    T = len(s)
    refractory_n = int(max(0, round(refractory_sec * fs)))

    events = []
    last_event_t = -10**9
    for t in range(1, T):
        if s[t-1] == 0 and s[t] == 1:
            if (t - last_event_t) >= refractory_n:
                events.append(t)
                last_event_t = t
    return len(events), np.array(events, dtype=np.int32)


def percentile_hysteresis_decode(y01: np.ndarray, q_enter: float = 0.80, q_exit: float = 0.60):
    """
    y01: (T,) in [0,1]
    enter threshold = quantile(y01, q_enter)
    exit  threshold = quantile(y01, q_exit)
    NOTE: q_enter > q_exit 권장 (히스테리시스 효과)
    """
    q_enter = float(np.clip(q_enter, 0.01, 0.99))
    q_exit  = float(np.clip(q_exit,  0.01, 0.99))

    thr_enter = float(np.quantile(y01, q_enter))
    thr_exit  = float(np.quantile(y01, q_exit))

    state = np.zeros_like(y01, dtype=np.int32)
    on = 0
    for i, v in enumerate(y01):
        if on == 0:
            if v >= thr_enter:
                on = 1
        else:
            if v <= thr_exit:
                on = 0
        state[i] = on
    return state.astype(np.float32), thr_enter, thr_exit


def decode_model_state_and_events(p_hat: np.ndarray, cfg):
    """
    Fixed post-processing for model output:
      (1) fixed smoothing on p_hat (no knob exposed)
      (2) percentile-based hysteresis on smoothed p_hat  <<< 변경
      (3) rising-edge + refractory counting
    Returns: (p_used_for_state, state01, pred_count, event_idx)
    """
    if p_hat is None:
        return None, None, 0, np.array([], dtype=np.int32)

    p_used = p_hat.astype(np.float32).copy()
    if len(p_used) >= 11:
        p_used = savgol_filter(p_used, window_length=11, polyorder=2).astype(np.float32)
        p_used = np.clip(p_used, 0.0, 1.0)

    # --- HERE: hysteresis -> percentile hysteresis ---
    state01, thr_enter, thr_exit = percentile_hysteresis_decode(
        p_used,
        q_enter=cfg["model_state_q_enter"],
        q_exit=cfg["model_state_q_exit"],
    )

    pred_count, event_idx = rising_edge_events_with_refractory(
        state01,
        fs=cfg["fs"],
        refractory_sec=cfg["count_refractory_sec"],
    )
    return p_used, state01, pred_count, event_idx


# =========================================================
# 3) Dataset (block-level fusion -> window slice)
# =========================================================
class MainEngineDataset(Dataset):
    def __init__(self, blocks, cfg):
        self.cfg = cfg
        self.groups = cfg["groups"]
        self.win = cfg["window_size"]
        self.stride = cfg["stride"]
        self.samples = []  # {"x":(win,C),"s":(win,),"y":(win,), "meta":{...}}

        for b in blocks:
            raw = b["raw"]
            subject = b["subject"]
            act = b["act"]
            T = len(raw)

            full_groups = {}
            for g in self.groups:
                Xg = get_group_array_from_block(raw, g)
                full_groups[g] = None if Xg is None else Xg.astype(np.float32)

            groups_dict = {}
            for g in self.groups:
                X_full = full_groups.get(g, None)
                if X_full is None:
                    continue
                c_full = compute_group_change(g, X_full)
                s_full = steady_score_from_change(c_full)
                q_full = estimate_self_consistency_quality(g, X_full, s_full)
                groups_dict[g] = {"X_full": X_full, "s_full": s_full, "q_full": float(q_full)}

            if len(groups_dict) == 0:
                continue

            s_fused_full, w_block, q_block = fuse_group_scores_block(groups_dict)
            y_full = (1.0 - s_fused_full).astype(np.float32)

            y_full = quantile_calibrate_01(y_full, q=cfg["calib_q"])
            s_fused_full = (1.0 - y_full).astype(np.float32)

            avail_groups = [g for g in self.groups if full_groups.get(g, None) is not None]

            for st in range(0, T - self.win + 1, self.stride):
                ed = st + self.win
                x_parts = [full_groups[g][st:ed] for g in avail_groups]
                if len(x_parts) == 0:
                    continue

                Xcat = np.concatenate(x_parts, axis=1).astype(np.float32)
                mu = Xcat.mean(axis=0, keepdims=True)
                sd = Xcat.std(axis=0, keepdims=True) + 1e-6
                Xcat = (Xcat - mu) / sd

                self.samples.append(
                    {
                        "x": Xcat,
                        "s": s_fused_full[st:ed].astype(np.float32),
                        "y": y_full[st:ed].astype(np.float32),
                        "meta": {"subject": subject, "act": act, "weights": w_block, "qualities": q_block},
                    }
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        d = self.samples[idx]
        x = torch.tensor(d["x"], dtype=torch.float32).transpose(0, 1)  # (C,T)
        s = torch.tensor(d["s"], dtype=torch.float32)                  # (T,)
        y = torch.tensor(d["y"], dtype=torch.float32)                  # (T,)
        return x, s, y


# =========================================================
# 4) Model
# =========================================================
class MainEngineNet(nn.Module):
    def __init__(self, input_ch: int, hidden_dim: int, latent_dim: int):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(input_ch, hidden_dim, kernel_size=7, padding=3),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
        )
        self.to_latent = nn.Sequential(
            nn.Conv1d(hidden_dim, latent_dim, kernel_size=1),
            nn.ReLU(),
        )
        self.trans_head = nn.Sequential(
            nn.Conv1d(latent_dim, 32, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(32, 1, kernel_size=1),
        )
        self.grav_proj = nn.Linear(latent_dim, 3)
        self.decoder = nn.Sequential(
            nn.Conv1d(latent_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, input_ch, kernel_size=3, padding=1),
        )

    def forward(self, x):
        h = self.backbone(x)
        z = self.to_latent(h)                  # (B,D,T)
        p = torch.sigmoid(self.trans_head(z))  # (B,1,T)
        x_recon = self.decoder(z)              # (B,C,T)
        return z, p, x_recon


# =========================================================
# 5) Losses (pair params fixed)
# =========================================================
def soft_bce(p_hat, y_soft, eps=1e-6):
    p = p_hat.squeeze(1)
    y = y_soft
    return F.binary_cross_entropy(p.clamp(eps, 1 - eps), y.clamp(eps, 1 - eps))


def pair_consistency_loss(z, s, delta: int = 5, margin: float = 0.2):
    B, D, T = z.shape
    if T <= delta:
        return torch.tensor(0.0, device=z.device)

    z1 = z[:, :, :-delta]
    z2 = z[:, :, delta:]
    s1 = s[:, :-delta]
    s2 = s[:, delta:]

    z1n = F.normalize(z1, dim=1)
    z2n = F.normalize(z2, dim=1)
    sim = (z1n * z2n).sum(dim=1)  # (B,T-d)

    w_pos = (s1 * s2).detach()
    w_neg = ((1 - s1) * (1 - s2)).detach()

    pos_loss = (w_pos * (1.0 - sim)).mean()
    neg_loss = (w_neg * F.relu(sim - margin)).mean()
    return pos_loss + neg_loss


def inertial_invariance_loss(z, x, s, model, eps=1e-6):
    B, D, T = z.shape
    C = x.shape[1]
    nvec = C // 3
    if nvec == 0:
        return torch.tensor(0.0, device=z.device)

    x_reshaped = x[:, : nvec * 3, :].reshape(B, nvec, 3, T)
    g = x_reshaped.mean(dim=3).mean(dim=1)  # (B,3)
    g = F.normalize(g, dim=1)

    w = s / (s.sum(dim=1, keepdim=True) + eps)
    z_bar = (z * w.unsqueeze(1)).sum(dim=2)  # (B,D)

    z3 = model.grav_proj(z_bar)
    z3 = F.normalize(z3, dim=1)
    corr = torch.abs(F.cosine_similarity(z3, g, dim=1)).mean()
    return corr


# =========================================================
# 6) Train / Eval
# =========================================================
def train_one_fold(model, loader, cfg):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])

    model.train()
    for _ in range(cfg["epochs"]):
        for x, s, y in loader:
            x = x.to(DEVICE)
            s = s.to(DEVICE)
            y = y.to(DEVICE)

            z, p, x_recon = model(x)

            loss_recon = F.mse_loss(x_recon, x)
            loss_soft = soft_bce(p, y)
            loss_pair = pair_consistency_loss(z, s, delta=5, margin=0.2)
            loss_inert = inertial_invariance_loss(z, x, s, model)

            loss = (
                cfg["lambda_recon"] * loss_recon
                + cfg["lambda_soft"] * loss_soft
                + cfg["lambda_pair"] * loss_pair
                + cfg["lambda_inertial"] * loss_inert
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        sched.step()


@torch.no_grad()
def eval_soft_mae_mse(model, loader):
    model.eval()
    all_p, all_y = [], []
    for x, _, y in loader:
        x = x.to(DEVICE)
        _, p, _ = model(x)
        all_p.append(p.squeeze(1).cpu().numpy())
        all_y.append(y.cpu().numpy())
    P = np.concatenate(all_p, axis=0)
    Y = np.concatenate(all_y, axis=0)
    mae = float(np.mean(np.abs(P - Y)))
    mse = float(np.mean((P - Y) ** 2))
    return mae, mse


# =========================================================
# 7) Reporting helpers (block inference)
# =========================================================
def build_full_groups_from_raw(raw_24: np.ndarray, cfg):
    full_groups = {}
    for g in cfg["groups"]:
        Xg = get_group_array_from_block(raw_24, g)
        full_groups[g] = None if Xg is None else Xg.astype(np.float32)
    return full_groups


def detector_fused_on_block(raw_24: np.ndarray, cfg):
    full_groups = build_full_groups_from_raw(raw_24, cfg)

    groups_dict = {}
    for g in cfg["groups"]:
        X_full = full_groups.get(g, None)
        if X_full is None:
            continue
        c_full = compute_group_change(g, X_full)
        s_full = steady_score_from_change(c_full)
        q_full = estimate_self_consistency_quality(g, X_full, s_full)
        groups_dict[g] = {"X_full": X_full, "s_full": s_full, "q_full": float(q_full)}

    s_fused, w_dict, q_dict = fuse_group_scores_block(groups_dict)
    y = (1.0 - s_fused).astype(np.float32)

    y = quantile_calibrate_01(y, q=cfg["calib_q"])
    s_fused = (1.0 - y).astype(np.float32)
    y_hyst = hysteresis_decode(y, enter=0.7, exit=0.3)
    return s_fused, y, y_hyst, w_dict, q_dict


@torch.no_grad()
def infer_p_hat_on_block(model, raw_24: np.ndarray, cfg):
    model.eval()
    win = cfg["window_size"]
    stride = cfg["stride"]
    T = len(raw_24)

    full_groups = build_full_groups_from_raw(raw_24, cfg)
    avail_groups = [g for g in cfg["groups"] if full_groups.get(g, None) is not None]
    if len(avail_groups) == 0:
        return None

    p_sum = np.zeros(T, dtype=np.float32)
    p_cnt = np.zeros(T, dtype=np.float32)

    for st in range(0, T - win + 1, stride):
        ed = st + win
        Xcat = np.concatenate([full_groups[g][st:ed] for g in avail_groups], axis=1).astype(np.float32)

        mu = Xcat.mean(axis=0, keepdims=True)
        sd = Xcat.std(axis=0, keepdims=True) + 1e-6
        Xcat = (Xcat - mu) / sd

        x = torch.tensor(Xcat, dtype=torch.float32).transpose(0, 1).unsqueeze(0).to(DEVICE)
        _, p, _ = model(x)
        pw = p.squeeze(0).squeeze(0).detach().cpu().numpy().astype(np.float32)

        p_sum[st:ed] += pw
        p_cnt[st:ed] += 1.0

    return p_sum / (p_cnt + 1e-8)


@torch.no_grad()
def infer_z_on_block(model, raw_24: np.ndarray, cfg):
    """
    Returns z_full: (T, D) overlap-added latent
    """
    model.eval()
    win = cfg["window_size"]
    stride = cfg["stride"]
    T = len(raw_24)

    full_groups = build_full_groups_from_raw(raw_24, cfg)
    avail_groups = [g for g in cfg["groups"] if full_groups.get(g, None) is not None]
    if len(avail_groups) == 0:
        return None

    z_sum = None
    z_cnt = np.zeros(T, dtype=np.float32)

    for st in range(0, T - win + 1, stride):
        ed = st + win
        Xcat = np.concatenate([full_groups[g][st:ed] for g in avail_groups], axis=1).astype(np.float32)

        mu = Xcat.mean(axis=0, keepdims=True)
        sd = Xcat.std(axis=0, keepdims=True) + 1e-6
        Xcat = (Xcat - mu) / sd

        x = torch.tensor(Xcat, dtype=torch.float32).transpose(0, 1).unsqueeze(0).to(DEVICE)
        z, _, _ = model(x)  # (1,D,win)
        z = z.squeeze(0).detach().cpu().numpy().astype(np.float32)  # (D,win)

        if z_sum is None:
            z_sum = np.zeros((z.shape[0], T), dtype=np.float32)

        z_sum[:, st:ed] += z
        z_cnt[st:ed] += 1.0

    z_full = z_sum / (z_cnt[None, :] + 1e-8)  # (D,T)
    return z_full.transpose(1, 0)  # (T,D)


def pick_representative_fold_by_count(df_act: pd.DataFrame):
    """
    representative fold: median absolute count error (ignore NaN)
    """
    if len(df_act) == 0:
        return None
    d = df_act.dropna(subset=["abs_err_count"]).copy()
    if len(d) == 0:
        return None
    med = d["abs_err_count"].median()
    idx = (d["abs_err_count"] - med).abs().idxmin()
    return d.loc[idx].to_dict()


# =========================================================
# 8) Plots (ONLY 2 requested)
# =========================================================
def _dominant_group_for_act(act: int):
    # requested mapping
    if act in [6, 7]:
        return "arm_acc"
    if act == 12:
        return "ankle_acc"
    return "arm_acc"


def _mag_from_group(raw_24: np.ndarray, group: str):
    Xg = get_group_array_from_block(raw_24, group)
    if Xg is None:
        return None
    return np.linalg.norm(Xg, axis=1).astype(np.float32)


def plot_1_dominant_raw_plus_pred(act: int, test_sub: int, raw_24: np.ndarray,
                                 p_hat: np.ndarray, state01: np.ndarray,
                                 event_idx: np.ndarray, cfg, save_path: str):
    fs = cfg["fs"]
    T_show = min(int(cfg["plot_sec"] * fs), len(raw_24))
    t = np.arange(T_show) / fs

    g = _dominant_group_for_act(act)
    mag = _mag_from_group(raw_24[:T_show], g)
    if mag is None:
        # fallback
        mag = _mag_from_group(raw_24[:T_show], "arm_acc")

    p = p_hat[:T_show] if p_hat is not None else None
    s = state01[:T_show]

    # events within show range
    ev = event_idx[event_idx < T_show] if event_idx is not None else np.array([], dtype=np.int32)

    plt.figure(figsize=(14, 7))

    ax1 = plt.subplot(2, 1, 1)
    ax1.plot(t, mag, label=f"{g}_mag (dominant raw)")
    ax1.set_title(f"Act{act} (test_sub={test_sub}) | dominant raw magnitude + transition prediction")
    ax1.grid(alpha=0.25)
    ax1.legend()

    ax2 = plt.subplot(2, 1, 2)
    if p is not None:
        ax2.plot(t, p, label="model p_hat(t)", alpha=0.85)
    ax2.plot(t, s, label="model state(t) (hysteresis)", alpha=0.95)
    # event markers
    for e in ev:
        ax2.axvline(e / fs, linewidth=1.0, alpha=0.35)
    ax2.set_ylim(-0.05, 1.05)
    ax2.grid(alpha=0.25)
    ax2.legend()
    ax2.set_xlabel("time (sec)")

    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


def pca2_numpy(Z: np.ndarray):
    """
    Z: (N,D)
    returns: (N,2)
    """
    Zc = Z - Z.mean(axis=0, keepdims=True)
    # SVD
    U, S, Vt = np.linalg.svd(Zc, full_matrices=False)
    W = Vt[:2].T  # (D,2)
    return Zc @ W


def plot_2_latent_separation(act: int, test_sub: int, z_full: np.ndarray,
                             state01: np.ndarray, cfg, save_path: str):
    """
    z_full: (T,D)
    state01: (T,) 0/1
    """
    if z_full is None or len(z_full) == 0:
        return

    # optional subsample to avoid too many points
    T = len(z_full)
    step = 1
    if T > 20000:
        step = int(np.ceil(T / 20000))

    Z = z_full[::step]
    S = state01[::step]
    X2 = pca2_numpy(Z)

    steady = (S < 0.5)
    trans = (S >= 0.5)

    plt.figure(figsize=(8, 7))
    plt.scatter(X2[steady, 0], X2[steady, 1], s=6, alpha=0.45, label="steady (state=0)")
    plt.scatter(X2[trans, 0], X2[trans, 1], s=6, alpha=0.45, label="transition (state=1)")
    plt.title(f"Act{act} (test_sub={test_sub}) | latent z(t) separation (PCA2D)")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


# =========================================================
# 9) Main
# =========================================================
def main():
    print("=" * 70)
    print("Main Engine (MIN-HYPERPARAM) - MHEALTH LOSO (per-activity) + Event Counting")
    print("=" * 70)

    df_all = load_mhealth_df(CONFIG["data_dir"], CONFIG["target_activities"])
    blocks_all = create_blocks_by_subject_activity(df_all, CONFIG["window_size"])
    acts = sorted({b["act"] for b in blocks_all})

    records = []
    rep_artifacts = {}  # per act, store fold artifacts for representative plotting

    for act in acts:
        print("\n" + "=" * 70)
        print(f"[Activity {act}] Single-activity LOSO")
        print("=" * 70)

        blocks = [b for b in blocks_all if b["act"] == act]
        subjects = sorted({b["subject"] for b in blocks})
        print(f"[Blocks] act={act} total={len(blocks)} | folds={len(subjects)}")

        for test_sub in subjects:
            train_blocks = [b for b in blocks if b["subject"] != test_sub]
            test_blocks = [b for b in blocks if b["subject"] == test_sub]
            if len(train_blocks) == 0 or len(test_blocks) == 0:
                continue

            train_ds = MainEngineDataset(train_blocks, CONFIG)
            test_ds = MainEngineDataset(test_blocks, CONFIG)
            if len(train_ds) == 0 or len(test_ds) == 0:
                continue

            train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, drop_last=True)
            test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False)

            x0, _, _ = train_ds[0]
            input_ch = x0.shape[0]

            model = MainEngineNet(
                input_ch=input_ch,
                hidden_dim=CONFIG["hidden_dim"],
                latent_dim=CONFIG["latent_dim"],
            ).to(DEVICE)

            train_one_fold(model, train_loader, CONFIG)
            mae_soft, mse_soft = eval_soft_mae_mse(model, test_loader)

            # ----------------------------
            # Count evaluation on the test subject block (raw0)
            # ----------------------------
            b0 = test_blocks[0]
            raw0 = b0["raw"]

            p_hat = infer_p_hat_on_block(model, raw0, CONFIG)  # (T,)
            if p_hat is None:
                continue

            # fixed decode for model p_hat -> state/events
            p_used, state_model, pred_count, event_idx = decode_model_state_and_events(p_hat, CONFIG)

            gt_key = f"subject{test_sub}"
            gt_count = GT_COUNT.get(act, {}).get(gt_key, None)

            if gt_count is None:
                # If GT is missing, still log pred
                abs_err = np.nan
                signed_err = np.nan
            else:
                abs_err = float(abs(pred_count - gt_count))
                signed_err = float(pred_count - gt_count)

            records.append(
                {
                    "act": int(act),
                    "test_sub": int(test_sub),
                    "mae_soft": float(mae_soft),
                    "mse_soft": float(mse_soft),
                    "pred_count": int(pred_count),
                    "gt_count": (int(gt_count) if gt_count is not None else np.nan),
                    "abs_err_count": abs_err,
                    "err_count": signed_err,
                    "refractory_sec": float(CONFIG["count_refractory_sec"]),
                    "model_enter_q": float(CONFIG["model_state_q_enter"]),
                    "model_exit_q": float(CONFIG["model_state_q_exit"]),
                    "input_ch": int(input_ch),
                    "calib_q": float(CONFIG["calib_q"]),
                }
            )

            print(
                f"  Fold test_sub={test_sub:2d} | Soft(MAE={mae_soft:.4f}, MSE={mse_soft:.4f}) "
                f"| Count(pred={pred_count}, gt={gt_count}, err={signed_err:+.0f})"
            )

            # save for representative plotting
            rep_artifacts.setdefault(act, [])
            rep_artifacts[act].append(
                {
                    "test_sub": test_sub,
                    "abs_err_count": abs_err,
                    "raw0": raw0,
                    "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                    "input_ch": input_ch,
                }
            )

    df_res = pd.DataFrame(records)
    csv_path = os.path.join(CONFIG["out_dir"], "results_loso.csv")
    df_res.to_csv(csv_path, index=False)
    print("\n[Saved]", csv_path)

    # =====================================================
    # Representative plots (2 plots per activity, test subject)
    # =====================================================
    for act in acts:
        df_act = df_res[df_res["act"] == act].reset_index(drop=True)
        if len(df_act) == 0 or act not in rep_artifacts:
            continue

        rep = pick_representative_fold_by_count(df_act)
        if rep is None:
            continue
        rep_sub = int(rep["test_sub"])

        cand = None
        for item in rep_artifacts[act]:
            if int(item["test_sub"]) == rep_sub:
                cand = item
                break
        if cand is None:
            continue

        model = MainEngineNet(
            input_ch=int(cand["input_ch"]),
            hidden_dim=CONFIG["hidden_dim"],
            latent_dim=CONFIG["latent_dim"],
        ).to(DEVICE)
        model.load_state_dict(cand["model_state"], strict=True)
        model.eval()

        raw0 = cand["raw0"]

        # infer p_hat/state/events and z
        p_hat = infer_p_hat_on_block(model, raw0, CONFIG)
        p_used, state_model, pred_count, event_idx = decode_model_state_and_events(p_hat, CONFIG)
        z_full = infer_z_on_block(model, raw0, CONFIG)

        # Plot 1: dominant raw + prediction + events
        p1_path = os.path.join(CONFIG["out_dir"], f"act{act}_P1_raw_plus_pred_sub{rep_sub}.png")
        plot_1_dominant_raw_plus_pred(
            act=int(act),
            test_sub=int(rep_sub),
            raw_24=raw0,
            p_hat=p_hat,              # plot original p_hat(t)
            state01=state_model,      # state from fixed decode
            event_idx=event_idx,
            cfg=CONFIG,
            save_path=p1_path,
        )

        # Plot 2: latent separation
        p2_path = os.path.join(CONFIG["out_dir"], f"act{act}_P2_latent_separation_sub{rep_sub}.png")
        plot_2_latent_separation(
            act=int(act),
            test_sub=int(rep_sub),
            z_full=z_full,
            state01=state_model,
            cfg=CONFIG,
            save_path=p2_path,
        )

        print(f"[Saved plots] act={act} rep_sub={rep_sub} -> {CONFIG['out_dir']} "
              f"(pred_count={pred_count})")

    # =====================================================
    # Summary print
    # =====================================================
    if len(df_res) > 0:
        print("\n" + "=" * 70)
        print("Count summary (per activity):")
        print("=" * 70)
        for act in sorted(df_res["act"].unique()):
            dfa = df_res[df_res["act"] == act]
            # only rows with GT
            dfa2 = dfa.dropna(subset=["gt_count"])
            if len(dfa2) == 0:
                continue
            mae = float(np.mean(np.abs(dfa2["pred_count"].values - dfa2["gt_count"].values)))
            bias = float(np.mean(dfa2["pred_count"].values - dfa2["gt_count"].values))
            print(f"  act={act}: Count-MAE={mae:.3f} | Bias={bias:+.3f} | n={len(dfa2)}")

    print("\nDone.")


if __name__ == "__main__":
    main()


Main Engine (MIN-HYPERPARAM) - MHEALTH LOSO (per-activity) + Event Counting
[Data] Total samples: 68098 | Subjects: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)] | Acts: [np.int64(6), np.int64(7), np.int64(12)]

[Activity 6] Single-activity LOSO
[Blocks] act=6 total=10 | folds=10
  Fold test_sub= 1 | Soft(MAE=0.2771, MSE=0.1115) | Count(pred=23, gt=21, err=+2)
  Fold test_sub= 2 | Soft(MAE=0.2659, MSE=0.1040) | Count(pred=24, gt=19, err=+5)
  Fold test_sub= 3 | Soft(MAE=0.2683, MSE=0.1034) | Count(pred=26, gt=21, err=+5)
  Fold test_sub= 4 | Soft(MAE=0.2398, MSE=0.0906) | Count(pred=20, gt=20, err=+0)
  Fold test_sub= 5 | Soft(MAE=0.2412, MSE=0.0898) | Count(pred=18, gt=20, err=-2)
  Fold test_sub= 6 | Soft(MAE=0.2598, MSE=0.0965) | Count(pred=16, gt=20, err=-4)
  Fold test_sub= 7 | Soft(MAE=0.2654, MSE=0.1002) | Count(pred=23, gt=20, err=+3)
  Fold test_sub= 8 | Soft(MAE=0.2643, MSE=0.1058) | Count(p